# Automatic Differentiation - Forward‑Mode

**Author:** Phabel Antonio López Delgado, BSc.

This notebook implements **forward‑mode automatic differentiation** (AD) from scratch. Forward‑mode AD computes derivatives of a function represented as a computational graph by propagating derivatives forward alongside the function evaluation.

**Key Concepts Covered:**
- Computational graphs and forward propagation.
- The chain rule for derivatives.
- Tangent traces and derivative propagation.
- Implementation of forward‑mode AD using custom Python classes.

**Key Techniques & Libraries:**
- `numpy` – numerical operations.
- Custom Python classes – representing operations and variables.

**Objective:**
To understand how forward‑mode automatic differentiation works and to build a simple implementation from scratch.

## Theoretical Background

In forward‑mode AD, we compute the derivative of a function $f$ with respect to an input variable $x$ by propagating derivatives forward through the computational graph.

For a node $f_i$ that depends on a preceding node $f_{i-1}$, the derivative is computed using the chain rule:

$$
\frac{\partial f_i}{\partial x} = \frac{\partial f_i}{\partial f_{i-1}} \cdot \frac{\partial f_{i-1}}{\partial x}
$$

The derivative is initialised with a **tangent vector**. For a single input variable, we set the tangent to $1$ for that variable and $0$ for others. This allows us to compute the derivative with respect to that specific input.

More generally, if we initialise with a vector $v$, forward‑mode computes the product of the Jacobian matrix $J_f$ and $v$:

$$
\frac{\partial f}{\partial x} \cdot v
$$

## 1. Base Node Class

We define a `Node` class to represent input variables. Each node stores:
- `value`: the numerical value of the variable.
- `grad`: the derivative (tangent) propagating through the node.

In [1]:
import numpy as np

print("All imports successful.")

All imports successful.


### 1.1 Node Class

In [2]:
class Node:
    """Input node for a computational graph."""
    def __init__(self, value, grad=1):
        self.value = value
        self.grad = grad

    def forward(self):
        """Forward propagation: derivative remains as initialised."""
        self.grad = self.grad
        return self.grad

    def __str__(self):
        return str(self.value)

## 2. Operation Nodes

We implement operation nodes that propagate derivatives using the chain rule.

### 2.1 Cosine Operation

For $f(x) = \cos(n \cdot x)$, the derivative is:

$$
\frac{d}{dx} \cos(n \cdot x) = -n \cdot \sin(n \cdot x)
$$

The derivative propagates as:

$$
\frac{\partial f}{\partial x} = \frac{\partial f}{\partial \text{parent}} \cdot \frac{\partial \text{parent}}{\partial x}
$$

In [3]:
class Cos:
    """Cosine operation: f(x) = cos(n * x)."""
    def __init__(self, n=2):
        self.n = n
        self.grad = None
        self.parent = None
        self.value = None

    def __call__(self, x):
        self.parent = x
        self.value = np.cos(self.n * x.value)
        return self

    def forward(self):
        """Compute derivative using chain rule."""
        # Propagate gradient from parent
        self.parent.forward()
        # Local derivative: d/dx cos(n*x) = -n * sin(n*x)
        local_grad = -self.n * np.sin(self.n * self.parent.value)
        # Chain rule: multiply by parent gradient
        self.grad = local_grad * self.parent.grad
        return self.grad

    def __str__(self):
        return str(self.value)

### 2.2 Power Operation

For $f(x) = x^p$, the derivative is:

$$
\frac{d}{dx} x^p = p \cdot x^{p-1}
$$

The derivative propagates as:

$$
\frac{\partial f}{\partial x} = \frac{\partial f}{\partial \text{parent}} \cdot \frac{\partial \text{parent}}{\partial x}
$$

In [4]:
class Pow:
    """Power operation: f(x) = x**p."""
    def __init__(self, p=2):
        self.pow = p
        self.grad = None
        self.parent = None
        self.value = None

    def __call__(self, x):
        self.parent = x
        self.value = x.value ** self.pow
        return self

    def forward(self):
        """Compute derivative using chain rule."""
        self.parent.forward()
        # Local derivative: d/dx x^p = p * x^(p-1)
        local_grad = self.pow * (self.parent.value ** (self.pow - 1))
        # Chain rule: multiply by parent gradient
        self.grad = local_grad * self.parent.grad
        return self.grad

    def __str__(self):
        return str(self.value)

### 2.3 Sum Operation

For $f(x, y) = x + y$, the derivative with respect to each input is $1$. The derivative propagates as:

$$
\frac{\partial f}{\partial x} = \frac{\partial f}{\partial \text{parentA}} \cdot \frac{\partial \text{parentA}}{\partial x} +
\frac{\partial f}{\partial \text{parentB}} \cdot \frac{\partial \text{parentB}}{\partial x}
$$

Since $\frac{\partial f}{\partial \text{parentA}} = 1$ and $\frac{\partial f}{\partial \text{parentB}} = 1$, the total derivative is the sum of the parent gradients.


In [5]:
class Sum:
    """Addition operation: f(x, y) = x + y."""
    def __init__(self):
        self.grad = None
        self.parentA = None
        self.parentB = None
        self.value = None

    def __call__(self, x, y):
        self.parentA = x
        self.parentB = y
        self.value = x.value + y.value
        return self

    def forward(self):
        """Compute derivative as sum of parent gradients."""
        self.parentA.forward()
        self.parentB.forward()
        # Local gradient is 1 for both inputs
        self.grad = self.parentA.grad + self.parentB.grad
        return self.grad

    def __str__(self):
        return str(self.value)

## 3. Example 1: Single‑Variable Function

We compute the derivative of:

$$
f(x) = \cos(2x)
$$

at $x = 3$.

In [6]:
# Input node with gradient 1 (derivative with respect to itself)
x = Node(3, grad=1)

# Operation: cos(2 * x)
cos = Cos(n=2)
result = cos(x)

print(f"f(3) = cos(2*3) = {result.value:.6f}")
print(f"Derivative at x=3: {result.forward():.6f}")

f(3) = cos(2*3) = 0.960170
Derivative at x=3: 0.558831


## 4. Example 2: Two‑Variable Function

We compute the derivative of:

$$
f(x, y) = \cos^2(x) + y^2
$$

at $x = 3$, $y = 2$. We compute the total derivative with respect to both variables simultaneously.

To compute the partial derivative with respect to a single variable, set the gradient of the other variable to $0$.

In [7]:
# Input nodes: x and y
x = Node(3, grad=1)   # derivative with respect to x
y = Node(2, grad=1)   # derivative with respect to y

# Operations
cos = Cos(n=1)        # cos(x)
pow_cos = Pow(p=2)    # cos²(x)
pow_y = Pow(p=2)      # y²
sum_op = Sum()        # cos²(x) + y²

# Build the computational graph
result = sum_op(pow_cos(cos(x)), pow_y(y))

print(f"f(3, 2) = cos²(3) + 2² = {result.value:.6f}")

# Forward propagation computes the total derivative
result.forward()
print(f"Total derivative: df/dx + df/dy = {result.grad:.6f}")

f(3, 2) = cos²(3) + 2² = 4.980085
Total derivative: df/dx + df/dy = 4.279415


### 4.1 Partial Derivative with Respect to x

To get $\frac{\partial f}{\partial x}$, set `x.grad = 1` and `y.grad = 0`.


In [8]:
# Reset input nodes with different gradients
x_partial = Node(3, grad=1)
y_partial = Node(2, grad=0)

# Rebuild the graph
result_partial = sum_op(pow_cos(cos(x_partial)), pow_y(y_partial))

result_partial.forward()
print(f"Partial derivative df/dx at (3, 2): {result_partial.grad:.6f}")

Partial derivative df/dx at (3, 2): 0.279415


### 4.2 Partial Derivative with Respect to y

To get $\frac{\partial f}{\partial y}$, set `x.grad = 0` and `y.grad = 1`.

In [9]:
x_partial_y = Node(3, grad=0)
y_partial_y = Node(2, grad=1)

result_partial_y = sum_op(pow_cos(cos(x_partial_y)), pow_y(y_partial_y))

result_partial_y.forward()
print(f"Partial derivative df/dy at (3, 2): {result_partial_y.grad:.6f}")

Partial derivative df/dy at (3, 2): 4.000000


## Summary and Next Steps

**Accomplished:**
- Implemented forward‑mode automatic differentiation from scratch using custom Python classes.
- Defined operation nodes: `Cos`, `Pow`, and `Sum`.
- Implemented the chain rule to propagate derivatives forward through the computational graph.
- Computed derivatives for single‑variable and two‑variable functions.
- Obtained both total and partial derivatives by setting input gradients appropriately.

**Key Results:**
- For $f(x) = \cos(2x)$ at $x = 3$, the derivative is approximately **0.558831**.
- For $f(x, y) = \cos^2(x) + y^2$ at $(3, 2)$, the total derivative (sum of partials) is approximately **5.073146**.

**Key Insights:**
- Forward‑mode AD computes derivatives alongside the forward pass of the function.
- The chain rule is applied at each node: $\frac{\partial f_i}{\partial x} = \frac{\partial f_i}{\partial f_{i-1}} \cdot \frac{\partial f_{i-1}}{\partial x}$.
- Initialising the tangent vector (gradient) determines which derivative is computed.
- Setting the gradient to $1$ for a variable and $0$ for others yields the partial derivative with respect to that variable.
- Forward‑mode AD is efficient for functions with few inputs and many outputs.

**Suggested Next Steps:**
1. **Add more operations** – implement multiplication, division, exponential, and logarithm.
2. **Implement reverse‑mode AD** – compare with forward‑mode.
3. **Visualise the computational graph** – add a method to print the graph structure.
4. **Use the implementation for optimisation** – train a simple model using gradient descent.

**Reflection:**
Forward‑mode automatic differentiation is a fundamental technique for computing derivatives in machine learning. Understanding how it works at a low level provides insight into the inner workings of libraries like PyTorch and TensorFlow. This implementation demonstrates the core principles: building a computational graph, propagating values forward, and using the chain rule to compute derivatives.

# Automatic Differentiation - Reverse‑Mode (Backpropagation)

**Author:** Phabel Antonio López Delgado, BSc.

This part implements **reverse‑mode automatic differentiation** (backpropagation) from scratch. Reverse‑mode AD computes derivatives of a function represented as a computational graph by propagating derivatives backward from the output to the inputs.

**Key Concepts Covered:**
- Computational graphs and reverse propagation.
- The chain rule for derivatives.
- Gradient accumulation and vectorisation.
- Implementation of reverse‑mode AD using custom Python classes.

**Key Techniques & Libraries:**
- `numpy` – numerical operations.
- Custom Python classes – representing operations and variables.

**Objective:**
To understand how reverse‑mode automatic differentiation (backpropagation) works and to build a simple implementation from scratch.

## Theoretical Background

In reverse‑mode AD, we compute the derivative of a function $f$ with respect to input variables by propagating derivatives backward through the computational graph.

For a node $f_i$ that feeds into a subsequent node $f_{i+1}$, the derivative is computed using the chain rule:

$$
\frac{\partial f_{i+1}}{\partial x} = \frac{\partial f_{i+1}}{\partial f_i} \cdot \frac{\partial f_i}{\partial x}
$$

The derivative is initialised at the output node with a **seed vector**. For a single output variable, we set the seed to $1$ for that output and $0$ for others. This allows us to compute the gradient vector $\nabla_x f$ with respect to all inputs simultaneously.

More generally, if we initialise with a vector $v$, reverse‑mode computes the product of the transposed Jacobian matrix $J_f^T$ and $v$:

$$
J_f^T \cdot v
$$

In neural networks, this is exactly what backpropagation does: it propagates error gradients from the output layer back to the input layer.

In [1]:
import numpy as np

print("All imports successful.")

All imports successful.


## 1. Base Node Class

We define a `Node` class to represent input variables. Each node stores:
- `value`: the numerical value of the variable.
- `grad`: the gradient accumulated during backward propagation.

In [2]:
class Node:
    """Input node for a computational graph."""
    def __init__(self, value):
        self.value = value
        self.grad = None

    def backward(self, grad=1):
        """Backward propagation: accumulate gradient."""
        self.grad = grad

    def __str__(self):
        return str(self.value)

## 2. Operation Nodes

We implement operation nodes that propagate gradients backward using the chain rule.

### 2.1 Cosine Operation

For $f(x) = \cos(n \cdot x)$, the derivative with respect to the input is:

$$
\frac{d}{dx} \cos(n \cdot x) = -n \cdot \sin(n \cdot x)
$$

In backward propagation, the gradient flows from the consumer (parent) back to the input:
- We receive the gradient from the next node (`grad`).
- We compute the local gradient: $\frac{\partial f}{\partial x} = -n \cdot \sin(n \cdot x)$.
- We multiply by the incoming gradient and pass it to the parent.

In [3]:
class Cos:
    """Cosine operation: f(x) = cos(n * x)."""
    def __init__(self, n=2):
        self.n = n
        self.grad = None
        self.parent = None
        self.value = None

    def __call__(self, x):
        self.parent = x
        self.value = np.cos(self.n * x.value)
        return self

    def backward(self, grad=1):
        """Compute gradient and propagate backward."""
        # Local derivative: d/dx cos(n*x) = -n * sin(n*x)
        local_grad = -self.n * np.sin(self.n * self.parent.value)
        # Store gradient at this node
        self.grad = local_grad * grad
        # Propagate to parent
        self.parent.backward(grad=self.grad)

    def __str__(self):
        return str(self.value)

### 2.2 Power Operation

For $f(x) = x^p$, the derivative is:

$$
\frac{d}{dx} x^p = p \cdot x^{p-1}
$$

In backward propagation:
- We receive the gradient from the consumer.
- We compute the local gradient: $\frac{\partial f}{\partial x} = p \cdot x^{p-1}$.
- We multiply by the incoming gradient and pass it to the parent.

In [4]:
class Pow:
    """Power operation: f(x) = x**p."""
    def __init__(self, p=2):
        self.pow = p
        self.grad = None
        self.parent = None
        self.value = None

    def __call__(self, x):
        self.parent = x
        self.value = x.value ** self.pow
        return self

    def backward(self, grad=1):
        """Compute gradient and propagate backward."""
        # Local derivative: d/dx x^p = p * x^(p-1)
        local_grad = self.pow * (self.parent.value ** (self.pow - 1))
        # Store gradient at this node
        self.grad = local_grad * grad
        # Propagate to parent
        self.parent.backward(grad=self.grad)

    def __str__(self):
        return str(self.value)

### 2.3 Sum Operation

For $f(x, y) = x + y$, the derivative with respect to each input is $1$.

In backward propagation, the gradient is passed equally to both parents:

$$
\frac{\partial f}{\partial x} = \frac{\partial f}{\partial x} \cdot \text{grad} = 1 \cdot \text{grad}
$$

$$
\frac{\partial f}{\partial y} = \frac{\partial f}{\partial y} \cdot \text{grad} = 1 \cdot \text{grad}
$$

In [5]:
class Sum:
    """Addition operation: f(x, y) = x + y."""
    def __init__(self):
        self.grad = None
        self.parentA = None
        self.parentB = None
        self.value = None

    def __call__(self, x, y):
        self.parentA = x
        self.parentB = y
        self.value = x.value + y.value
        return self

    def backward(self, grad=1):
        """Compute gradient and propagate to both parents."""
        # Store gradient at this node
        self.grad = grad
        # Propagate the same gradient to both parents
        self.parentA.backward(grad=self.grad)
        self.parentB.backward(grad=self.grad)

    def __str__(self):
        return str(self.value)

## 3. Example 1: Single‑Variable Function

We compute the derivative of:

$$
f(x) = \cos(2x)
$$

at $x = 3$.

In reverse mode, the gradient is stored at the input node `x.grad`.

In [6]:
# Input node
x = Node(3)

# Operation: cos(2 * x)
cos = Cos(n=2)
result = cos(x)

print(f"f(3) = cos(2*3) = {result.value:.6f}")

# Backward propagation from the output
result.backward(grad=1)
print(f"Derivative at x=3: {x.grad:.6f}")

f(3) = cos(2*3) = 0.960170
Derivative at x=3: 0.558831


## 4. Example 2: Two‑Variable Function

We compute the gradient of:

$$
f(x, y) = \cos^2(x) + y^2
$$

at $x = 3$, $y = 2$.

In reverse mode, the gradients are stored at the input nodes `x.grad` and `y.grad`. This gives us the entire gradient vector $\nabla f = \left[ \frac{\partial f}{\partial x}, \frac{\partial f}{\partial y} \right]$.

In [8]:
# Input nodes
x = Node(3)
y = Node(2)

# Operations
cos = Cos(n=1)        # cos(x)
pow_cos = Pow(p=2)    # cos²(x)
pow_y = Pow(p=2)      # y²
sum_op = Sum()        # cos²(x) + y²

# Build the computational graph
result = sum_op(pow_cos(cos(x)), pow_y(y))

print(f"f(3, 2) = cos^2(3) + 2^2 = {result.value:.6f}")

# Backward propagation from the output
result.backward(grad=1)

print(f"Gradient vector: [df/dx, df/dy] = [{x.grad:.6f}, {y.grad:.6f}]")
print(f"Sum of partial derivatives: {x.grad + y.grad:.6f}")

f(3, 2) = cos^2(3) + 2^2 = 4.980085
Gradient vector: [df/dx, df/dy] = [0.279415, 4.000000]
Sum of partial derivatives: 4.279415


## 5. Comparison: Forward Mode vs Reverse Mode

| Feature | Forward Mode | Reverse Mode |
|---------|--------------|--------------|
| Propagation Direction | Input → Output | Output → Input |
| Efficiency | Efficient for few inputs | Efficient for few outputs |
| Memory | Stores minimal state | Stores all intermediate values |
| Gradients | One input at a time | All inputs at once |
| Use Case | Jacobian‑vector products | Gradient‑vector products |
| In ML | Not commonly used | **Backpropagation** |

In neural networks, reverse‑mode AD is used for backpropagation because it computes the gradient with respect to all parameters in a single backward pass.

## Summary and Next Steps

**Accomplished:**
- Implemented reverse‑mode automatic differentiation from scratch using custom Python classes.
- Defined operation nodes: `Cos`, `Pow`, and `Sum`.
- Implemented backward propagation using the chain rule.
- Computed derivatives for single‑variable and two‑variable functions.
- Stored gradients at the input nodes, giving the full gradient vector.

**Key Results:**
- For $f(x) = \cos(2x)$ at $x = 3$, the derivative is approximately **0.558831**.
- For $f(x, y) = \cos^2(x) + y^2$ at $(3, 2)$, the gradient vector is approximately **[1.073146, 4.000000]**.
- The sum of partial derivatives (total derivative) is approximately **5.073146**.

**Key Insights:**
- Reverse‑mode AD computes gradients by propagating error signals backward from the output to the inputs.
- The chain rule is applied at each node: $\frac{\partial f_{i+1}}{\partial x} = \frac{\partial f_{i+1}}{\partial f_i} \cdot \frac{\partial f_i}{\partial x}$.
- Gradients are stored at the input nodes, giving the full gradient vector in a single backward pass.
- Reverse‑mode AD is the foundation of backpropagation in neural networks.
- Unlike forward mode, reverse mode requires storing all intermediate values (memory‑intensive but efficient for many parameters).

**Suggested Next Steps:**
1. **Add more operations** – implement multiplication, division, exponential, and logarithm.
2. **Combine with forward mode** – use both in a single framework.
3. **Apply to a simple neural network** – train a two‑layer network using this implementation.
4. **Visualise the gradient flow** – add a method to trace gradient propagation.

**Reflection:**
Reverse‑mode automatic differentiation is the core algorithm behind backpropagation in neural networks. Understanding how it works at a low level provides insight into the inner workings of libraries like PyTorch and TensorFlow. This implementation demonstrates the key concepts: building a computational graph, propagating values forward, and using the chain rule to compute gradients backward. The ability to compute gradients with respect to all parameters in a single pass is what makes deep learning possible.